# Statistics That Matter for Data Science

Practical statistical concepts that appear in every DS project:
**hypothesis testing**, the **bootstrap**, **power analysis**, and
**multiple comparisons correction**.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. Sampling distributions and the Central Limit Theorem

The mean of $n$ independent draws converges to a Normal distribution
regardless of the population distribution — this underpins almost every
standard test.

In [ ]:
rng = np.random.default_rng(42)
n_samples, n_obs = 5000, [1, 5, 30, 200]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, n in zip(axes, n_obs):
    means = [rng.exponential(scale=1, size=n).mean() for _ in range(n_samples)]
    ax.hist(means, bins=50, color=C[0], alpha=0.85, density=True, edgecolor="white", linewidth=0.3)
    ax.axvline(np.mean(means), color=C[1], lw=2, linestyle="--", label=f"mean={np.mean(means):.2f}")
    ax.set(title=f"n = {n}", xlabel="Sample mean", ylabel="Density" if n==1 else "")
    ax.legend(fontsize=9)
fig.suptitle("CLT: sample means of Exponential(1) converge to Normal as n grows",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 2. Bootstrap confidence intervals

Model-free uncertainty estimation — no distributional assumptions,
works on any statistic (median, ratio, AUC).

In [ ]:
rng   = np.random.default_rng(42)
data  = rng.lognormal(mean=4, sigma=0.8, size=250)   # skewed revenue data
n_boot = 10_000

boot_means   = [rng.choice(data, size=len(data), replace=True).mean()  for _ in range(n_boot)]
boot_medians = [np.median(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, stat, col in zip(axes,
        [boot_means, boot_medians], ["Mean", "Median"], [C[0], C[2]]):
    lo, hi = np.percentile(vals, [2.5, 97.5])
    ax.hist(vals, bins=60, color=col, alpha=0.85, density=True,
            edgecolor="white", linewidth=0.3)
    ax.axvline(lo, color="black", lw=1.8, linestyle="--", label=f"95% CI [{lo:.0f}, {hi:.0f}]")
    ax.axvline(hi, color="black", lw=1.8, linestyle="--")
    ax.axvline(np.mean(vals), color=C[1], lw=2, label=f"Point estimate: {np.mean(vals):.0f}")
    ax.set(title=f"Bootstrap {stat} (n=250, log-normal revenue)",
           xlabel=stat, ylabel="Density")
    ax.legend()
plt.tight_layout()
plt.show()
print(f"Mean    95% CI: [{np.percentile(boot_means,2.5):.1f}, {np.percentile(boot_means,97.5):.1f}]")
print(f"Median  95% CI: [{np.percentile(boot_medians,2.5):.1f}, {np.percentile(boot_medians,97.5):.1f}]")
print("Median CI is narrower -> median is more efficient for log-normal data")

## 3. Power analysis — how many samples?

Underpowered studies produce inflated estimates *when* they happen to
reach significance (winner's curse). Run power analysis *before* the
experiment.

In [ ]:
from statsmodels.stats.power import TTestIndPower

analysis  = TTestIndPower()
baseline  = 0.10
lifts_pp  = np.arange(0.5, 5.1, 0.5)   # effect sizes in percentage points
alphas    = [0.01, 0.05, 0.10]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: required n vs effect size
for alpha, col in zip(alphas, [C[1], C[0], C[2]]):
    ns = []
    for lift in lifts_pp:
        p_treat   = baseline + lift / 100
        sigma     = ((baseline*(1-baseline) + p_treat*(1-p_treat)) / 2)**0.5
        effect_d  = (lift / 100) / sigma
        n         = analysis.solve_power(effect_size=effect_d, alpha=alpha, power=0.80)
        ns.append(n)
    ax1.plot(lifts_pp, ns, "o-", color=col, label=f"alpha={alpha}")
ax1.set(xlabel="True lift (percentage points)", ylabel="n per arm (power=80%)",
        title="Sample size vs lift size
(baseline conversion=10%)", yscale="log")
ax1.legend()
ax1.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x,_: f"{int(x):,}"))

# Panel 2: power vs n for a 2pp lift
lift_pp = 2.0
p_treat = baseline + lift_pp / 100
sigma   = ((baseline*(1-baseline) + p_treat*(1-p_treat)) / 2)**0.5
eff_d   = (lift_pp / 100) / sigma
ns2     = np.linspace(500, 20_000, 200)
for alpha, col in zip(alphas, [C[1], C[0], C[2]]):
    powers = [analysis.solve_power(effect_size=eff_d, nobs1=n, alpha=alpha, power=None)
              for n in ns2]
    ax2.plot(ns2, powers, color=col, label=f"alpha={alpha}")
ax2.axhline(0.80, color="grey", lw=1.5, linestyle="--", label="Power=80%")
ax2.axhline(0.90, color="grey", lw=1.0, linestyle=":")
ax2.set(xlabel="n per arm", ylabel="Statistical power",
        title=f"Power vs n for +{lift_pp}pp lift
(baseline={baseline:.0%})")
ax2.legend()
plt.tight_layout()
plt.show()

## 4. Multiple comparisons — Benjamini-Hochberg correction

Running 20 tests at alpha=5% gives ~1 false positive in expectation.
The BH correction controls the **False Discovery Rate** (FDR) rather
than the per-test alpha — more powerful than Bonferroni.

In [ ]:
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(42)
n_tests   = 30
n_true    = 5   # truly significant hypotheses

p_true   = rng.uniform(0, 0.005, n_true)    # small p (true effects)
p_noise  = rng.uniform(0.05, 1.0, n_tests - n_true)  # null hypotheses
p_values = np.concatenate([p_true, p_noise])
is_true  = np.array([True]*n_true + [False]*(n_tests-n_true))
rng.shuffle(p_values)   # mix them up (shuffle doesn't track truth labels — demo only)

_, reject_bonf, _, _ = multipletests(p_values, alpha=0.05, method="bonferroni")
_, reject_bh,   _, _ = multipletests(p_values, alpha=0.05, method="fdr_bh")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: p-value distribution
ax = axes[0]
ax.scatter(range(n_tests), sorted(p_values), color=C[0], s=50, zorder=3, label="p-value")
ax.axhline(0.05, color=C[1], lw=1.8, linestyle="--", label="Raw alpha=0.05")
bonf_thresh = 0.05 / n_tests
ax.axhline(bonf_thresh, color=C[3], lw=1.8, linestyle="-.", label=f"Bonferroni={bonf_thresh:.4f}")
# BH threshold line
sorted_p = np.sort(p_values)
bh_thresholds = 0.05 * (np.arange(1, n_tests+1)) / n_tests
bh_line_idx = np.where(sorted_p <= bh_thresholds)[0]
if len(bh_line_idx):
    ax.axhline(sorted_p[bh_line_idx[-1]], color=C[2], lw=1.8, linestyle=":", label=f"BH adaptive threshold")
ax.set(xlabel="Test rank", ylabel="p-value",
       title="p-values: raw vs corrected thresholds")
ax.legend(fontsize=9)

# Right: rejection comparison
ax = axes[1]
methods = ["Raw (p<0.05)", "Bonferroni", "Benjamini-Hochberg"]
n_reject = [(p_values < 0.05).sum(), reject_bonf.sum(), reject_bh.sum()]
bars = ax.bar(methods, n_reject, color=[C[1], C[3], C[2]], width=0.5)
ax.bar_label(bars, padding=3, fontsize=11, fontweight="bold")
ax.set(ylabel="Rejections", title="Number of rejected hypotheses
(5 are truly significant)",
       ylim=(0, max(n_reject)+3))
ax.axhline(n_true, color="black", lw=1.5, linestyle="--", label=f"True positives = {n_true}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Raw:    {(p_values < 0.05).sum()} rejections (inflation risk)")
print(f"Bonf:   {reject_bonf.sum()} rejections (too conservative)")
print(f"BH:     {reject_bh.sum()} rejections (controls FDR at 5%)")